# LightGBM — Walmart Store Sales Forecasting

**WandB Project:** `walmart-sales-forecasting-project`  
**Runs:** Cleaning → Feature Engineering → Baseline → Tuned v1 → Tuned v2 → CV → Best Pipeline  
**Metric:** WMAE — სადღესასწაულო კვირა იწონება 5x-ით

In [ ]:
# ბიბლიოთეკების იმპორტი
import pandas as pd
import numpy as np
import lightgbm as lgb
import wandb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print('ბიბლიოთეკები ჩაიტვირთა')

In [ ]:
# WandB-ის კონფიგურაცია
WANDB_PROJECT = 'walmart-sales-forecasting-project'
WANDB_ENTITY  = 'ashos22-free-university-of-tbilisi-'

wandb.login()
print('WandB login OK')

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

## 1. მონაცემების ჩატვირთვა

In [ ]:
DATA_PATH = '/kaggle/input/walmart-recruiting-store-sales-forecasting/'

train    = pd.read_csv(DATA_PATH + 'train.csv')
test     = pd.read_csv(DATA_PATH + 'test.csv')
stores   = pd.read_csv(DATA_PATH + 'stores.csv')
features = pd.read_csv(DATA_PATH + 'features.csv')

print('train:   ', train.shape)
print('test:    ', test.shape)
print('stores:  ', stores.shape)
print('features:', features.shape)
train.head()

## 2. WandB Run — LightGBM_Cleaning

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Cleaning',
    group='LightGBM_Training',
    reinit=True
)


for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

# stores და features-ის მიბმა
train = train.merge(stores, on='Store', how='left')
train = train.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))
test  = test.merge(stores, on='Store', how='left')
test  = test.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))


for df in [train, test]:
    if 'IsHoliday_feat' in df.columns:
        df.drop('IsHoliday_feat', axis=1, inplace=True)


markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
for df in [train, test]:
    df[markdown_cols] = df[markdown_cols].fillna(0)


for col in ['CPI', 'Unemployment']:
    train[col] = train.groupby('Store')[col].transform(lambda x: x.ffill())
    test[col]  = test.groupby('Store')[col].transform(lambda x: x.ffill())


type_map = {'A': 0, 'B': 1, 'C': 2}
train['Type'] = train['Type'].map(type_map)
test['Type']  = test['Type'].map(type_map)


train['IsHoliday'] = train['IsHoliday'].astype(int)
test['IsHoliday']  = test['IsHoliday'].astype(int)

wandb.log({
    'train_rows':  len(train),
    'test_rows':   len(test),
    'null_train':  int(train.isnull().sum().sum()),
    'null_test':   int(test.isnull().sum().sum())
})
wandb.config.update({'train_shape': str(train.shape), 'test_shape': str(test.shape)})
run.finish()

print('გასუფთავება დასრულდა')
print('Null train:', train.isnull().sum().sum())
print('Null test: ', test.isnull().sum().sum())

## 3. WandB Run — LightGBM_Feature_Engineering

In [ ]:
FEATURE_COLS = [
    'Store', 'Dept', 'Type', 'Size', 'IsHoliday',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'Quarter',
    'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_26', 'lag_52',
    'rolling_mean_4', 'rolling_mean_8', 'rolling_mean_13',
    'rolling_std_4',  'rolling_std_8',  'rolling_std_13'
]

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Feature_Engineering',
    group='LightGBM_Training',
    reinit=True
)

train_tmp = train[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
test_tmp  = test[['Store', 'Dept', 'Date']].copy()
test_tmp['Weekly_Sales'] = np.nan

combined = pd.concat([train_tmp, test_tmp]).sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)


for lag in [1, 2, 4, 8, 26, 52]:
    combined[f'lag_{lag}'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(lag)


for window in [4, 8, 13]:
    combined[f'rolling_mean_{window}'] = (
        combined.groupby(['Store', 'Dept'])['Weekly_Sales']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    combined[f'rolling_std_{window}'] = (
        combined.groupby(['Store', 'Dept'])['Weekly_Sales']
        .transform(lambda x: x.shift(1).rolling(window).std())
    )


combined['Year']    = combined['Date'].dt.year
combined['Month']   = combined['Date'].dt.month
combined['Week']    = combined['Date'].dt.isocalendar().week.astype(int)
combined['Quarter'] = combined['Date'].dt.quarter


train_meta = train[['Store', 'Dept', 'Date', 'Type', 'Size', 'IsHoliday',
                     'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                     'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']].drop_duplicates()
test_meta  = test[['Store', 'Dept', 'Date', 'Type', 'Size', 'IsHoliday',
                    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']].drop_duplicates()
all_meta   = pd.concat([train_meta, test_meta]).drop_duplicates(subset=['Store', 'Dept', 'Date'])

combined = combined.merge(all_meta, on=['Store', 'Dept', 'Date'], how='left')


train_fe = combined[combined['Weekly_Sales'].notna()].copy()
train_fe = train_fe.dropna(subset=['lag_1'])  # lag-ების null-ების წაშლა

test_fe  = combined[combined['Weekly_Sales'].isna()].drop('Weekly_Sales', axis=1).copy()
for col in ['rolling_std_4', 'rolling_std_8', 'rolling_std_13']:
    test_fe[col] = test_fe[col].fillna(0)

wandb.config.update({
    'feature_count':      len(FEATURE_COLS),
    'train_rows_after_fe': len(train_fe),
    'test_rows':           len(test_fe),
    'lag_values':          [1, 2, 4, 8, 26, 52],
    'rolling_windows':     [4, 8, 13]
})
run.finish()

print(f'Feature Engineering დასრულდა')
print(f'Train: {len(train_fe):,} | Test: {len(test_fe):,} | Features: {len(FEATURE_COLS)}')

## 4. Train / Validation Split

დროის მიხედვით ვყოფთ — random split დაუშვებელია Time Series-ისთვის.

In [ ]:
CUTOFF = pd.Timestamp('2012-08-01')

tr = train_fe[train_fe['Date'] < CUTOFF]
vl = train_fe[train_fe['Date'] >= CUTOFF]

X_tr, y_tr, h_tr = tr[FEATURE_COLS], tr['Weekly_Sales'], tr['IsHoliday']
X_vl, y_vl, h_vl = vl[FEATURE_COLS], vl['Weekly_Sales'], vl['IsHoliday']

print(f'Train:      {len(tr):,} | {tr["Date"].min().date()} → {tr["Date"].max().date()}')
print(f'Validation: {len(vl):,} | {vl["Date"].min().date()} → {vl["Date"].max().date()}')

## 5. WandB Run — LightGBM_Baseline

In [ ]:
params_baseline = {
    'objective':     'regression',
    'metric':        'mae',
    'n_estimators':  500,
    'learning_rate': 0.1,
    'num_leaves':    31,
    'verbose':       -1,
    'random_state':  42
}

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Baseline',
    group='LightGBM_Training',
    config=params_baseline,
    reinit=True
)

model_baseline = lgb.LGBMRegressor(**params_baseline)
model_baseline.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

preds_baseline = model_baseline.predict(X_vl)
score_baseline = wmae(y_vl.values, preds_baseline, h_vl.values)

wandb.log({'wmae_val': score_baseline, 'best_iteration': model_baseline.best_iteration_})
run.finish()

print(f'Baseline WMAE: {score_baseline:.4f}')

## 6. WandB Run — LightGBM_Tuned_v1

In [ ]:
params_v1 = {
    'objective':         'regression',
    'metric':            'mae',
    'n_estimators':      2000,
    'learning_rate':     0.05,
    'num_leaves':        127,
    'max_depth':         8,
    'min_child_samples': 50,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':           -1,
    'random_state':      42
}

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Tuned_v1',
    group='LightGBM_Training',
    config=params_v1,
    reinit=True
)

model_v1 = lgb.LGBMRegressor(**params_v1)
model_v1.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)]
)

preds_v1 = model_v1.predict(X_vl)
score_v1 = wmae(y_vl.values, preds_v1, h_vl.values)


fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': model_v1.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi.head(20), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance (Top 20)')
plt.tight_layout()

wandb.log({
    'wmae_val':        score_v1,
    'best_iteration':  model_v1.best_iteration_,
    'feature_importance': wandb.Image(fig)
})
plt.show()
run.finish()

print(f'Tuned v1 WMAE: {score_v1:.4f}  (Baseline: {score_baseline:.4f})')

## 7. WandB Run — LightGBM_Tuned_v2

Huber loss — outlier-ების მიმართ მდგრადი.

In [ ]:
params_v2 = {
    'objective':         'huber',
    'alpha':             0.9,
    'metric':            'mae',
    'n_estimators':      2000,
    'learning_rate':     0.03,
    'num_leaves':        255,
    'max_depth':         10,
    'min_child_samples': 30,
    'subsample':         0.7,
    'colsample_bytree':  0.7,
    'reg_alpha':         0.05,
    'reg_lambda':        0.5,
    'verbose':           -1,
    'random_state':      42
}

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Tuned_v2',
    group='LightGBM_Training',
    config=params_v2,
    reinit=True
)

model_v2 = lgb.LGBMRegressor(**params_v2)
model_v2.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)]
)

preds_v2 = model_v2.predict(X_vl)
score_v2 = wmae(y_vl.values, preds_v2, h_vl.values)

wandb.log({'wmae_val': score_v2, 'best_iteration': model_v2.best_iteration_})
run.finish()

print(f'Tuned v2 WMAE: {score_v2:.4f}')

## 8. WandB Run — LightGBM_CV

3-fold expanding window cross-validation.

In [ ]:

best_params_cv = params_v1 if score_v1 <= score_v2 else params_v2

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_CV',
    group='LightGBM_Training',
    config=best_params_cv,
    reinit=True
)

all_dates = sorted(train_fe['Date'].unique())
n_folds   = 3
fold_size = len(all_dates) // (n_folds + 1)
cv_scores = []

for fold in range(n_folds):
    cutoff      = all_dates[fold_size * (fold + 1)]
    val_end     = all_dates[min(fold_size * (fold + 2), len(all_dates) - 1)]

    tr_fold = train_fe[train_fe['Date'] <  cutoff]
    vl_fold = train_fe[(train_fe['Date'] >= cutoff) & (train_fe['Date'] < val_end)]

    if len(vl_fold) == 0:
        continue

    m = lgb.LGBMRegressor(**best_params_cv)
    m.fit(tr_fold[FEATURE_COLS], tr_fold['Weekly_Sales'],
          eval_set=[(vl_fold[FEATURE_COLS], vl_fold['Weekly_Sales'])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    score_fold = wmae(vl_fold['Weekly_Sales'].values,
                      m.predict(vl_fold[FEATURE_COLS]),
                      vl_fold['IsHoliday'].values)
    cv_scores.append(score_fold)
    wandb.log({f'wmae_fold_{fold+1}': score_fold})
    print(f'Fold {fold+1}: {score_fold:.4f}')

cv_mean = np.mean(cv_scores)
cv_std  = np.std(cv_scores)
wandb.log({'wmae_cv_mean': cv_mean, 'wmae_cv_std': cv_std})
run.finish()

print(f'\nCV Mean: {cv_mean:.4f} ± {cv_std:.4f}')

## 9. საუკეთესო მოდელი → WandB Artifact

In [ ]:
scores = {'Baseline': score_baseline, 'Tuned_v1': score_v1, 'Tuned_v2': score_v2}
best_name  = min(scores, key=scores.get)
best_score = scores[best_name]
best_model = {'Baseline': model_baseline, 'Tuned_v1': model_v1, 'Tuned_v2': model_v2}[best_name]
best_params = {'Baseline': params_baseline, 'Tuned_v1': params_v1, 'Tuned_v2': params_v2}[best_name]

print('შედეგები:')
for k, v in scores.items():
    marker = ' ← საუკეთესო' if k == best_name else ''
    print(f'  {k}: {v:.4f}{marker}')

In [ ]:
import joblib

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='LightGBM_Best_Pipeline',
    group='LightGBM_Training',
    reinit=True
)

# სრულ train-ზე გადატრენინგება საუკეთესო პარამეტრებით
final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(train_fe[FEATURE_COLS], train_fe['Weekly_Sales'])

# მოდელის ფაილად შენახვა
joblib.dump(final_model, 'lightgbm_model.pkl')

# WandB Artifact-ად ატვირთვა
artifact = wandb.Artifact(
    name='lightgbm-walmart-sales',
    type='model',
    metadata={'wmae_val': best_score, 'best_version': best_name, 'feature_cols': FEATURE_COLS}
)
artifact.add_file('lightgbm_model.pkl')
run.log_artifact(artifact)

wandb.log({'wmae_val_best': best_score})
run.finish()

print(f'WandB Artifact: lightgbm-walmart-sales')
print(f'საუკეთესო WMAE: {best_score:.4f} ({best_name})')